In [2]:
# Standard library
import os
import gc, json
import time
import warnings

from collections import Counter

# Numerical computing and data processing
import numpy as np
import pandas as pd
import scipy

from pathlib import Path

import numpy as np
import librosa
import soundfile as sf

from soundfile import LibsndfileError

# Visualization
import matplotlib.pyplot as plt
from PIL import Image


# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights

# UMAP
import umap
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# K-means reproducibility: number of random initializations per run
KMEANS_N_INIT = 100

from pathlib import Path

DATA_ROOT = 'dataset'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Merging Datasets

For the next milestones in this project, we have decided to merge the 30-second mel spectrograms from the GTZAN and MTG-Jamendo Datasets. The aim of this technique is both to incorporate a wide range of training examples and to reduce our algorithms' bias towards selection decisions in any individual dataset. In order to prevent issues with class imbalance and to reduce the size of the dataset for efficient model training and deployment, at present we have only included genres that are attested in both datasets, with at least 500 samples in which said genre is the primary genre in MTG-Jamendo. This leaves us with 7 genres and a high proportion of possibly multi-genre songs, making this a useful selection for training.

We then converted all of the mel spectrogram data into image form. Using the functions below, we cut the width of all MTG-Jamendo samples to match 30-second samples, and we scaled the mel spectrograms from the .WAV forms of the files in the GTZAN dataset to match these in dimensionality. This will allow us to train models on both sets of data.

Load and standardize genre tags (simplified form of code above)

In [3]:
def convert_path(pth):
    return f"/mnt/WDR1/melspecs/{pth.split('.')[0]}.npy"

genre_tags = pd.read_excel('raw_30s_cleantags.xlsx')

genre_tags = genre_tags.fillna("_")

genre_tags.head()

missing = []

for pth in genre_tags['PATH']:
    if not os.path.exists(convert_path(pth)):
        missing.append(pth)

all_genres = []

for i, row in genre_tags.iterrows():
    genre_list = []
    for n in range(1, 46, 1):
        if row[f'TAGS_{n}'].startswith('genre'):
            genre_list.append(row[f'TAGS_{n}'].split('genre---')[1])
    
    all_genres.append(genre_list)

genre_tags['GENRE'] = all_genres
is_multigenre = [int(len(l) > 1) for l in all_genres]
genre_tags['MULTIGENRE'] = is_multigenre

empty_genres = [i for i, g in enumerate(all_genres) if len(g) == 0]

#Cut rows with no genre tags -- these are not useable for our analysis
genre_tags = genre_tags.drop(index=empty_genres)

#Identify primary genre in MTG dataset

primary_genre = [a[0] for a in genre_tags['GENRE']]
genre_tags["PRIMARY_GENRE"] = primary_genre

clean_genre_tags = genre_tags.drop(columns=[f"TAGS_{n}" for n in range(1, 46, 1)])

#Get and print primary genre counts
pg_counts = clean_genre_tags['PRIMARY_GENRE'].value_counts()
pg_counts = pg_counts[pg_counts > 500]
print(pg_counts)
pg_counts.shape

PRIMARY_GENRE
ambient          7149
electronic       6928
classical        4427
alternative      3663
easylistening    2864
pop              2530
chillout         2345
dance            2128
folk             1361
hiphop           1261
blues             874
soundtrack        780
indie             772
rock              735
jazz              683
atmospheric       623
funk              604
experimental      603
orchestral        531
poprock           511
metal             501
Name: count, dtype: int64


(21,)

In [4]:
shared_genres = [a.strip("/ \\") for a in os.listdir('/home/giod/Documents/DS II/Project/dataset/Data/genres_original') if a.strip("/ \\") in pg_counts.index.to_list()]

print(shared_genres)

['jazz', 'metal', 'pop', 'hiphop', 'classical', 'blues', 'rock']


In [5]:
def wav_to_mel_spectrogram_image(
    wav_path: str | Path,
    out_path: str | Path,
    clip_duration_sec: float = 30.0,
    target_sr: int = 22050,
    n_fft: int = 2048,
    hop_length: int = 512,
    n_mels: int = 256,
    fmin: float = 0.0,
    fmax: float | None = None,
    width_px: int = 1220,
    height_px: int = 900,
    dpi: int = 200,
    vmin: float = -80.0,
    vmax: float = 0.0,
) -> None:
    """
    Convert a .wav file to a mel spectrogram image.

    Output style:
    - magma colormap
    - no axes
    - no borders
    - default size 1135 x 852 px

    Uses the first `clip_duration_sec` seconds of audio.
    Pads with zeros if the file is shorter than that.
    """
    wav_path = Path(wav_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Load audio as mono
    y, sr = sf.read(str(wav_path), always_2d=True)
    y = y.mean(axis=1).astype(np.float32)

    # Resample if needed
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # Trim or pad to exactly clip_duration_sec
    target_len = int(round(clip_duration_sec * sr))
    if len(y) >= target_len:
        y = y[:target_len]
    else:
        padded = np.zeros(target_len, dtype=np.float32)
        padded[:len(y)] = y
        y = padded

    # Mel spectrogram -> dB
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Convert target pixels to matplotlib figsize
    fig_w = width_px / dpi
    fig_h = height_px / dpi

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])  # fill whole figure, no margins

    ax.imshow(
        mel_db,
        origin="lower",
        aspect="auto",
        cmap="magma",
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_axis_off()

    fig.savefig(
        out_path,
        dpi=dpi,
        format=out_path.suffix.lstrip(".") or "png",
        bbox_inches=None,
        pad_inches=0,
    )
    plt.close(fig)


In [7]:
final_mtg_data = []

for g in shared_genres:
    mtg_selection = clean_genre_tags[clean_genre_tags['PRIMARY_GENRE'] == g].sample(500, random_state=109)
    final_mtg_data.append(mtg_selection)

    directory = f'dataset_merged_small/{g}'
    os.makedirs(directory, exist_ok=True)

    dpi = 200
    fig_w = 610 / dpi
    fig_h = 450 / dpi

    for p in mtg_selection['PATH']:
        mtg_num = p.split('.')[0].split('/')[-1]
        mtg_ms_fm = f"{directory}/mtg_{mtg_num}.png"
        if not os.path.exists(mtg_ms_fm):
            mtg_melspec = np.load(convert_path(p))
            
            if mtg_melspec.shape[1] > 1407: #1407 represents min width of a 30' clip
                mtg_melspec = mtg_melspec[:, :1407]
            
            # plt.imshow(mtg_melspec, aspect="auto", origin="lower", cmap="magma")  # change cmap here
            # plt.tight_layout()
            # plt.axis("off")  # ← hides axes, ticks, and frame
            # plt.savefig(mtg_ms_fm, dpi=200, bbox_inches="tight", pad_inches=0)

            plt.figure(figsize=(fig_w, fig_h), dpi=dpi)

            plt.imshow(mtg_melspec, aspect="auto", origin="lower", cmap="magma")
            plt.axis("off")

            plt.savefig(mtg_ms_fm, dpi=dpi, bbox_inches="tight", pad_inches=0)
            plt.close()

    for gtz_fn in os.listdir(f'/home/giod/Documents/DS II/Project/dataset/Data/genres_original/{g}'):

        new_fn = f'{directory}/gtz_{gtz_fn.split(".")[0]}_{gtz_fn.split(".")[1]}.png'

        if not os.path.exists(new_fn):

            try:
                wav_to_mel_spectrogram_image(
                    f'/home/giod/Documents/DS II/Project/dataset/Data/genres_original/{g}/{gtz_fn}',
                    new_fn,
                    width_px=472,
                    height_px=346
                )
            except LibsndfileError:
                continue

In [ ]:
final_mtg_selections = pd.concat(final_mtg_data)
final_mtg_selections.to_csv("MTG Info for Merging.csv")

In [ ]:
counts = final_mtg_selections['MULTIGENRE'].value_counts()

base = counts.copy()
extra = pd.Series(0, index=counts.index)
extra.loc[0] = 600  # only add to the multigenre bar

# Plot stacked bars
plt.bar(base.index, base.values, label='MTG')
plt.bar(extra.index, extra.values, bottom=base.values, color='red', label='GTZAN')

plt.xticks(ticks=base.index, labels=['multi-genre', 'single-genre'])
plt.ylabel("Count")
plt.title("Distribution of multigenre vs single-genre songs in the merged corpus")
plt.legend();

![a](merged_dist.png)